# Generic Android image → TWRP Device Tree

**Easy mode: upload the images and run. No filenames need to be typed.**

NEXT automatically finds conventional `recovery.img`, `boot.img`, `vendor_boot.img` and `init_boot.img` files, including `_a` / `_b` slot names, chooses the recovery-bearing primary image, adds companions when present, generates the tree, ZIPs it, and downloads the result.

This works with supported Android boot/recovery images from Samsung, Google, Xiaomi, OnePlus, Motorola, Sony, Nothing, ASUS, Fairphone and other OEMs. Samsung FUS is not required here.

In [ ]:
import os, subprocess, sys
repo = '/content/twrpdtgen-NEXT'
if not os.path.isdir(os.path.join(repo, '.git')):
    subprocess.run(['git', 'clone', '-q', 'https://github.com/RectifyData/twrpdtgen-NEXT.git', repo], check=True)
else:
    subprocess.run(['git', '-C', repo, 'pull', '--ff-only'], check=True)
os.chdir(repo)
subprocess.run(['bash', 'launcher.sh', '--setup-only'], check=True)
commit = subprocess.check_output(['git', 'rev-parse', '--short', 'HEAD'], text=True).strip()
next_version = subprocess.check_output([sys.executable, '-c', 'import twrpdtgen; print(twrpdtgen.__version__)'], text=True).strip()
print(f'twrpdtgen-NEXT {next_version} @ {commit} ready')


## Upload images

Upload the image files you have. You can select several at once. A ZIP containing `.img` files is also accepted and its images are extracted automatically.

You do **not** need to rename or type the detected filenames when they use standard Android names.

In [ ]:
from io import BytesIO
from pathlib import Path, PurePosixPath
import shutil, zipfile
from google.colab import files

input_dir = Path('/content/twrp-input')
shutil.rmtree(input_dir, ignore_errors=True)
input_dir.mkdir(parents=True)

uploaded = files.upload()
found = []
for uploaded_name, data in uploaded.items():
    safe_name = Path(uploaded_name).name
    if safe_name.lower().endswith('.zip') and zipfile.is_zipfile(BytesIO(data)):
        with zipfile.ZipFile(BytesIO(data)) as bundle:
            for info in bundle.infolist():
                if info.is_dir() or not info.filename.lower().endswith('.img'):
                    continue
                pure = PurePosixPath(info.filename)
                if pure.is_absolute() or '..' in pure.parts:
                    continue
                out = input_dir / pure.name
                if out.exists():
                    raise RuntimeError(f'Duplicate image filename in uploads: {out.name}')
                with bundle.open(info) as src, out.open('wb') as dst:
                    shutil.copyfileobj(src, dst, 1024 * 1024)
                found.append(out)
    else:
        out = input_dir / safe_name
        out.write_bytes(data)
        if out.suffix.lower() == '.img':
            found.append(out)

if not found:
    raise RuntimeError('No .img files were uploaded or found inside the ZIP.')
print('Images ready for auto-detection:')
for path in sorted(found):
    print(' ', path.name, f'({path.stat().st_size} bytes)')


## Optional advanced switches

The defaults are the recommended automatic path. You normally do not need to change anything here.

In [ ]:
skeleton = 'auto' #@param ['auto', 'omni', 'twrp-12.1', 'twrp-14.1']
recovery_layout = 'auto' #@param ['auto', 'dedicated', 'boot', 'vendor_boot']
recovery_root_mode = 'auto' #@param ['auto', 'minimal', 'full']
include_stock_root_in_zip = False #@param {type:'boolean'}


In [ ]:
from pathlib import Path
import shutil, subprocess, sys
from google.colab import files

tree_output = Path('/content/twrp-tree')
archive = Path('/content/twrp-device-tree.zip')
shutil.rmtree(tree_output, ignore_errors=True)
archive.unlink(missing_ok=True)

args = [
    sys.executable, '-m', 'twrpdtgen.auto_tree',
    '/content/twrp-input',
    '-o', str(tree_output),
    '--archive', str(archive),
    '--skeleton', skeleton,
    '--recovery-layout', recovery_layout,
    '--recovery-root-mode', recovery_root_mode,
    '--overwrite',
]
if include_stock_root_in_zip:
    args.append('--include-stock-root')

print('Auto-detecting image roles and generating...')
subprocess.run(args, check=True)
print(f'\nReady: {archive} ({archive.stat().st_size} bytes)')
files.download(str(archive))


## What gets detected automatically

NEXT recognizes standard recovery/boot image names and slot suffixes, prefers a dedicated recovery image when present, then uses `vendor_boot`, `boot`, or `init_boot` as appropriate and attaches the other discovered images as companions.

If two equally plausible images for the same role are present, NEXT stops instead of silently choosing the wrong firmware variant. For normal single-device uploads, there is no required text entry.